### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv('AviationData.csv', encoding='latin1', low_memory=False)

print(df.shape)
df.info()

(88889, 31)
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-nul

In [3]:
# Inspect missing values
df.isna().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date                    0
dtype: i

In [4]:
# Summary statistics for numeric columns
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

**Cleaning assumption:** `Event.Date` needs to be parsed as a real date so we can filter to the client's 40-year active-lifetime window (1983 onward, given the data runs through 2023). 

For `Aircraft.Category`, over 60% of rows are missing this field even after the date filter — historically the NTSB did not consistently tag this field before it was added to their schema, and the vast majority of *labeled* rows are 'Airplane'. Rather than drop this majority of the data, we exclude only the explicitly non-airplane categories (Helicopter, Glider, Balloon, etc.) and keep NaN rows (treating them as presumed fixed-wing airplanes, consistent with historical labeling practice). This is a documented assumption/limitation of the analysis.

We also filter to `Amateur.Built == 'No'`, since the client is only interested in professionally built aircraft.

In [5]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter to aircraft that entered/were still in service within the last 40 years of the dataset (1983 onward)
df = df[df['Event.Date'] >= '1983-01-01'].copy()
print('After date filter:', df.shape)

After date filter: (85289, 31)


In [6]:
df['Aircraft.Category'].value_counts(dropna=False)

Aircraft.Category
NaN                  56566
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [7]:
non_airplane_categories = ['Helicopter', 'Glider', 'Balloon', 'Gyrocraft', 'Weight-Shift',
                             'Powered Parachute', 'Ultralight', 'WSFT', 'Powered-Lift', 'Blimp',
                             'Rocket', 'ULTR', 'Unknown', 'UNK']

# Exclude explicit non-airplane categories; NaNs are kept (presumed fixed-wing airplane)
df = df[~df['Aircraft.Category'].isin(non_airplane_categories)]
print('After aircraft category filter:', df.shape)

After aircraft category filter: (81007, 31)


In [8]:
df['Amateur.Built'].value_counts(dropna=False)

Amateur.Built
No     73002
Yes     7906
NaN       99
Name: count, dtype: int64

In [9]:
# Client only wants professional builds
df = df[df['Amateur.Built'] != 'Yes']
print('After amateur-built filter:', df.shape)

After amateur-built filter: (73101, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Constructing the injury metric.** The four injury/outcome columns (`Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Minor.Injuries`, `Total.Uninjured`) have missing values that we treat as 0 (i.e., no injuries of that type were recorded) — this is a reasonable assumption since the NTSB tends to record a 0/blank when a category doesn't apply rather than leaving it as a true unknown.

We estimate **total occupants aboard** (`Total.People`) as the sum of all four columns. This lets us:
1. Compute a **fraction of occupants fatally/seriously injured** (`Serious.Fatal.Fraction`) — a rate that is comparable across flights/aircraft of different sizes, rather than raw injury counts which are naturally higher for larger aircraft.
2. Use `Total.People` as a rough proxy for aircraft passenger capacity, which we'll use later to separate small vs. large aircraft.

We drop the small number of rows where `Total.People` is 0, since we can't compute a meaningful injury fraction for a flight with no recorded occupants (this may reflect unoccupied/ground aircraft or incomplete records).

In [10]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols] = df[injury_cols].fillna(0)

df['Total.People'] = df[injury_cols].sum(axis=1)
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']

# Drop records where we have no occupant information at all
df = df[df['Total.People'] > 0]

df['Serious.Fatal.Fraction'] = df['Fatal.Serious.Injuries'] / df['Total.People']
print('After occupant filter:', df.shape)
df['Serious.Fatal.Fraction'].describe()

After occupant filter: (71886, 34)


count    71886.000000
mean         0.271393
std          0.427969
min          0.000000
25%          0.000000
50%          0.000000
75%          0.666667
max          1.000000
Name: Serious.Fatal.Fraction, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [11]:
df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    52346
Destroyed      14917
NaN             2354
Minor           2195
Unknown           74
Name: count, dtype: int64

**Cleaning assumption:** Rows where `Aircraft.damage` is missing or `'Unknown'` don't tell us anything about the outcome we're measuring, so we drop them. We then create a simple binary `Destroyed` column (1 = aircraft was a total loss, 0 = Substantial/Minor damage only) as our second key safety measure alongside the injury fraction.

In [12]:
df = df[df['Aircraft.damage'].notna() & (df['Aircraft.damage'] != 'Unknown')]
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
print('After damage filter:', df.shape)
df['Destroyed'].value_counts(normalize=True)

After damage filter: (69458, 35)


Destroyed
0    0.785237
1    0.214763
Name: proportion, dtype: float64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning tasks identified for `Make`:**
- Inconsistent casing — e.g. `'Cessna'`, `'CESSNA'`, and `'cessna'` are all the same manufacturer but currently counted separately.
- Leading/trailing whitespace.

We standardize casing/whitespace, then restrict our analysis to Makes with at least 50 records (post-cleaning), so that any per-Make statistics we compute later are based on a reasonably sized sample.

In [13]:
df['Make'].nunique()

1803

In [14]:
df['Make'] = df['Make'].str.strip().str.upper().str.title()
df['Make'].nunique()

1493

In [15]:
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index

df = df[df['Make'].isin(valid_makes)]
print('After make filter:', df.shape, '| number of makes retained:', df['Make'].nunique())

After make filter: (64216, 35) | number of makes retained: 81


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

**Cleaning tasks for `Model`:** drop the small number of rows with a missing model. Model numbers (e.g. `'172'`) are **not** unique across makes — many manufacturers reuse simple numeric model codes — so we build a combined `Make.Model` identifier to uniquely reference a specific plane type.

In [16]:
df = df[df['Model'].notna()]
df['Model'] = df['Model'].str.strip().str.upper()

df['Make.Model'] = df['Make'] + ' ' + df['Model']
print('After model filter:', df.shape, '| number of unique make/models:', df['Make.Model'].nunique())

After model filter: (64201, 36) | number of unique make/models: 5661


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

**Cleaning tasks for the remaining columns:**
- `Weather.Condition`: `'UNK'` and `'Unk'` both mean unknown — consolidate to `'Unknown'`; fill true NaNs the same way.
- `Engine.Type`: consolidate `'UNK'`, `'NONE'`, `'LR'` (a data-entry artifact) into `'Unknown'`; fill NaNs the same way.
- `Purpose.of.flight`, `Broad.phase.of.flight`: fill NaNs with `'Unknown'` rather than dropping — as the instructions note, we don't need to drop these NaNs since 'Unknown' is a legitimate category we can still visualize/compare against.
- `Number.of.Engines`: left as-is; NaNs here are relatively rare and we'll handle them at the point of use if needed.

In [17]:
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': 'Unknown', 'Unk': 'Unknown'}).fillna('Unknown')
df['Engine.Type'] = df['Engine.Type'].replace({'UNK': 'Unknown', 'NONE': 'Unknown', 'LR': 'Unknown'}).fillna('Unknown')
df['Purpose.of.flight'] = df['Purpose.of.flight'].fillna('Unknown')
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')

print(df['Weather.Condition'].value_counts())
print()
print(df['Engine.Type'].value_counts())

Weather.Condition
VMC        56690
IMC         4975
Unknown     2536
Name: count, dtype: int64

Engine.Type
Reciprocating      53582
Unknown             4382
Turbo Prop          2597
Turbo Shaft         2022
Turbo Fan           1243
Turbo Jet            374
Geared Turbofan        1
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

**Dropping sparse/irrelevant columns.** A number of columns are missing the large majority of their values, or aren't relevant to the safety questions the client is asking (location metadata, administrative/report fields). We drop these rather than trying to impute them, since imputing a column that's 60-90% missing wouldn't produce trustworthy values. We also drop `Aircraft.Category` and `Investigation.Type`, which we've already used for filtering and don't need going forward.

In [18]:
df.isna().sum().sort_values(ascending=False)

Schedule                  55661
Air.carrier               54241
FAR.Description           47272
Aircraft.Category         47197
Longitude                 41730
Latitude                  41725
Airport.Code              26681
Airport.Name              24861
Publication.Date          11400
Report.Status              2991
Number.of.Engines          2947
Registration.Number         848
Country                     178
Amateur.Built                49
Location                     35
Event.Id                      0
Event.Date                    0
Accident.Number               0
Investigation.Type            0
Injury.Severity               0
Engine.Type                   0
Model                         0
Make                          0
Aircraft.damage               0
Total.Serious.Injuries        0
Total.Minor.Injuries          0
Purpose.of.flight             0
Total.Fatal.Injuries          0
Weather.Condition             0
Total.Uninjured               0
Broad.phase.of.flight         0
Total.Pe

In [19]:
drop_cols = ['Event.Id', 'Accident.Number', 'Airport.Code', 'Airport.Name', 'Latitude', 'Longitude',
             'Registration.Number', 'Schedule', 'Air.carrier', 'FAR.Description', 'Report.Status',
             'Publication.Date', 'Aircraft.Category', 'Investigation.Type']

df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)
df.columns.tolist()

(64201, 22)


['Event.Date',
 'Location',
 'Country',
 'Injury.Severity',
 'Aircraft.damage',
 'Make',
 'Model',
 'Amateur.Built',
 'Number.of.Engines',
 'Engine.Type',
 'Purpose.of.flight',
 'Total.Fatal.Injuries',
 'Total.Serious.Injuries',
 'Total.Minor.Injuries',
 'Total.Uninjured',
 'Weather.Condition',
 'Broad.phase.of.flight',
 'Total.People',
 'Fatal.Serious.Injuries',
 'Serious.Fatal.Fraction',
 'Destroyed',
 'Make.Model']

In [20]:
df.isna().sum().sort_values(ascending=False)

Number.of.Engines         2947
Country                    178
Amateur.Built               49
Location                    35
Event.Date                   0
Injury.Severity              0
Make                         0
Aircraft.damage              0
Model                        0
Engine.Type                  0
Purpose.of.flight            0
Total.Fatal.Injuries         0
Total.Serious.Injuries       0
Total.Minor.Injuries         0
Total.Uninjured              0
Weather.Condition            0
Broad.phase.of.flight        0
Total.People                 0
Fatal.Serious.Injuries       0
Serious.Fatal.Fraction       0
Destroyed                    0
Make.Model                   0
dtype: int64

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [21]:
df.to_csv('cleaned_aviation_data.csv', index=False)
print('Saved cleaned_aviation_data.csv with shape', df.shape)

Saved cleaned_aviation_data.csv with shape (64201, 22)
